# Response Generation and Embedding Extraction

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
import json
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
BASE_PATH = '/content/drive/MyDrive/Project-4/'
HF_TOKEN = os.environ.get("HF_TOKEN")
MODEL_NAME = "meta-llama/Llama-3.2-3B"

BEST_LAYERS = {
    'afraid': 6, 'angry': 5, 'anxious': 9, 'devastated': 8,
    'lonely': 4, 'sad': 7, 'terrified': 1, 'furious': 7,
    'grateful': 15, 'hopeful': 12, 'faithful': 5
}

In [13]:
def analyze_similarity(best_embeddings):
    """Analyze cosine similarity with prototypes"""
    print("\n" + "="*60)
    print("COSINE SIMILARITY ANALYSIS")
    print("="*60)
    
    context_embs = np.load(BASE_PATH + 'context_embeddings.npz', allow_pickle=True)
    emotions = list(BEST_LAYERS.keys())
    results = []
    
    for key, response_emb in best_embeddings.items():
        context = key.rsplit('_', 1)[0]
        best_layer = BEST_LAYERS[context]
        response_emb = response_emb.reshape(1, -1)
        
        # Compare against all emotion prototypes
        sims = {}
        for emotion in emotions:
            proto = context_embs[f"{emotion}_{best_layer}"].reshape(1, -1)
            sims[emotion] = cosine_similarity(response_emb, proto)[0][0]
        
        predicted_emotion = max(sims, key=sims.get)
        
        results.append({
            'context': context,
            'correct_sim': sims[context],
            'predicted_emotion': predicted_emotion,
            'correct': (predicted_emotion == context),
            **{f"sim_{e}": sims[e] for e in emotions}
        })
    
    results_df = pd.DataFrame(results)
    results_df.to_csv(BASE_PATH + 'response_similarity_analysis.csv', index=False)
    
    # Print statistics
    accuracy_per_context = results_df.groupby('context')['correct'].mean()
    print("\nAccuracy per context:")
    for context in emotions:
        print(f"  {context:<12}: {accuracy_per_context[context]:.2%}")
    
    print(f"\nOverall accuracy: {results_df['correct'].mean():.2%}")
    print(f"✓ Saved to {BASE_PATH}response_similarity_analysis.csv")
    print("="*60)
    
    return results_df

In [14]:
# Load context embeddings from JSON and convert to NPZ
with open(BASE_PATH + 'context_embeddings.json', 'r') as f:
    context_data = json.load(f)

# Convert to NPZ format
context_embeddings = {}

for context, layers_dict in context_data.items():
    for layer_str, embedding in layers_dict.items():
        layer_idx = int(layer_str)
        key = f"{context}_{layer_idx}"
        context_embeddings[key] = np.array(embedding)

# Save as NPZ
np.savez_compressed(BASE_PATH + 'context_embeddings.npz', **context_embeddings)

print(f"✓ Converted context.json to context_embeddings.npz")
print(f"✓ Total prototypes: {len(context_embeddings)}")

✓ Converted context.json to context_embeddings.npz
✓ Total prototypes: 319


In [15]:
# Load existing best embeddings and run cosine similarity analysis
best_embeddings_npz = np.load(BASE_PATH + 'response_embeddings_best_layer.npz', allow_pickle=True)
best_embeddings = {key: best_embeddings_npz[key] for key in best_embeddings_npz.files}

# Analyze similarity
results_df = analyze_similarity(best_embeddings)


COSINE SIMILARITY ANALYSIS

Accuracy per context:
  afraid      : 2.00%
  angry       : 4.00%
  anxious     : 7.00%
  devastated  : 1.00%
  lonely      : 32.00%
  sad         : 35.00%
  terrified   : 13.00%
  furious     : 2.00%
  grateful    : 7.00%
  hopeful     : 22.00%
  faithful    : 9.00%

Overall accuracy: 12.18%
✓ Saved to /content/drive/MyDrive/Project-4/response_similarity_analysis.csv


In [ ]:
# This cell is no longer needed - delete it or keep for reference


COSINE SIMILARITY ANALYSIS


ValueError: Cannot load file containing pickled data when allow_pickle=False